In [230]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, IntegerType
import time

In [231]:

spark = SparkSession.builder.appName("AdvanceSparkApp").getOrCreate()

df = spark.range(0, 100000).withColumn("value", col("id") % 1000)


In [232]:
df.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [233]:
print("Before Partitions:", df.rdd.getNumPartitions())

Before Partitions: 2


In [234]:
df_repartitioned = df.repartition(10)

In [235]:
print("After Partitions:", df_repartitioned.rdd.getNumPartitions())



After Partitions: 10


[Stage 441:>                                                        (0 + 2) / 2]

In [236]:
df_coalesced = df_repartitioned.coalesce(2)

In [237]:
print("After Coalesced:", df_coalesced.rdd.getNumPartitions())


After Coalesced: 2


In [240]:
df_repartitioned = df.repartition()

TypeError: DataFrame.repartition() missing 1 required positional argument: 'numPartitions'

In [241]:
spark.stop()

In [242]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PartitionDemo").getOrCreate()

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data.csv")

26/06/13 07:10:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [243]:
print("Current Partitions:", df.rdd.getNumPartitions())

Current Partitions: 1


In [245]:
df_repartitioned = df.repartition(20)

print("After Repartition:", df_repartitioned.rdd.getNumPartitions())

After Repartition: 20


In [ ]:
partition_sizes = df_repartitioned.rdd.glom().map(len).collect()

print(partition_sizes)

[Stage 7:=======================================>                 (14 + 2) / 20]

In [247]:
df_repartitioned.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output")

In [40]:
spark.stop()

In [41]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, IntegerType
import time

In [42]:

spark = SparkSession.builder.appName("AdvanceSparkApp").getOrCreate()

df = spark.range(0, 100000).withColumn("value", col("id") % 1000)


26/06/13 05:47:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [43]:
df.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [209]:
 ## optimization and Caching 
optimized_df.unpersist()

DataFrame[id: bigint, value: bigint]

In [98]:
optimized_df =df.filter(col("value") >500).filter(col("id")<5000000).select("id","value")

In [99]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#331L, (id#331L % 1000) AS value#333L]
+- *(1) Filter (((id#331L % 1000) > 500) AND (id#331L < 5000000))
   +- *(1) Range (0, 100000, step=1, splits=2)




In [100]:
start_time =time.time()
count_uncached=optimized_df.count() # ACTION TRIGGER DAG
end_time=time.time()
print(f"1. Optimized Execution | Count:{count_uncached} |Time: {round(end_time - start_time,4)}seconds")

1. Optimized Execution | Count:49900 |Time: 0.0656seconds


In [101]:
optimized_df.cache()

DataFrame[id: bigint, value: bigint]

In [214]:

start_time =time.time()
count_cached=optimized_df.count() # ACTION TRIGGER DAG
end_time=time.time()
print(f"1. second  Optimized Execution | Count:{count_cached} |Time: {round(end_time - start_time,4)}seconds")

1. second  Optimized Execution | Count:49900 |Time: 0.0353seconds


In [215]:
data = [("Alice", 25), ("Bob", 17), ("Charlie", 35), ("David", 15)]
df1 = spark.createDataFrame(data, ["Name", "Age"])

In [216]:
df1.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 17|
|Charlie| 35|
|  David| 15|
+-------+---+



In [217]:
def categorize_age(age):
    if age >= 18:
        return "Adult"
    return "Minor"

In [218]:
age_category_udf = udf(categorize_age, StringType())

In [219]:
df_method1 = df1.withColumn("Category", age_category_udf(col("Age")))
print("Method 1: Standard UDF via DataFrame API")
df_method1.show()

Method 1: Standard UDF via DataFrame API


+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 35|   Adult|
|  David| 15|   Minor|
+-------+---+--------+



In [220]:
spark.udf.register("sql_categorize_age", categorize_age, StringType())

df1.createOrReplaceTempView("people")

In [221]:
sql_df = spark.sql("""
    SELECT
       Name,
       Age, 
       sql_categorize_age(Age) AS Category
   FROM people
""")

sql_df.show()

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 35|   Adult|
|  David| 15|   Minor|
+-------+---+--------+



In [225]:
def can_drive(age):
      if age >= 18:
        return "Yes"
      return "No"
    

In [226]:
can_drive_udf=udf(can_drive, StringType())

In [229]:
df_drive =df1.withColumn(
    "Can DRIVE",
    can_drive_udf(col("age"))
)
df_drive.show()

+-------+---+---------+
|   Name|Age|Can DRIVE|
+-------+---+---------+
|  Alice| 25|      Yes|
|    Bob| 17|       No|
|Charlie| 35|      Yes|
|  David| 15|       No|
+-------+---+---------+

